# Setup

In [ ]:
%%capture
# 1. Install dependencies
%pip install --quiet pydantic-ai sentence-transformers numpy mcp nest_asyncio openai
# 1b. Patch asyncio for Jupyter (REQUIRED — run this before any agent call)
import nest_asyncio
nest_asyncio.apply()


In [ ]:
# 2. Set your OpenRouter API key
# (Get one at https://openrouter.ai — gives access to 300+ models with one key.)
#
# Paste your key here, OR leave it as None to be prompted interactively,
# OR set it in your shell as the env var OPENROUTER_API_KEY.
OPENROUTER_API_KEY = ''    # e.g. 'sk-or-v1-...'

import os, getpass

if OPENROUTER_API_KEY:
    os.environ['OPENROUTER_API_KEY'] = OPENROUTER_API_KEY
elif 'OPENROUTER_API_KEY' not in os.environ:
    os.environ['OPENROUTER_API_KEY'] = getpass.getpass('OpenRouter API key: ')

In [25]:
import httpx
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

MODEL = OpenAIChatModel(
    'anthropic/claude-haiku-4.5',
    provider=OpenAIProvider(
        base_url='https://openrouter.ai/api/v1',
        api_key=os.environ['OPENROUTER_API_KEY'],
        http_client=httpx.AsyncClient(timeout=60),
    ),
)

In [26]:
def pretty_print_trace(result):
    for i, msg in enumerate(result.all_messages()):
        print(f'\n[{i}] {type(msg).__name__}')
        for part in getattr(msg, "parts", []):
            kind = type(part).__name__
            snippet = repr(part)[:200]
            print(f'    └─ {kind}: {snippet}')

# Validators

In [27]:
from pydantic_ai import Agent
from pydantic import BaseModel, Field

class PaperSummary(BaseModel):
    authors: list[str] | None = Field(description="List of Names and Surnames of the authors")
    achieved_accuracy: float | None = Field(description="Achieved accuracy in percentage")

abstract = "In this work authored by John and Jane, we have achieved accuracy of 74%"

agent = Agent(model=MODEL, 
              system_prompt="Extract relevant fields from the provided abstract.", 
              output_type=PaperSummary)

result = agent.run_sync(f"Abstract to parse: {abstract}")
result.output

PaperSummary(authors=['John', 'Jane'], achieved_accuracy=74.0)

In [29]:
from pydantic import field_validator

class ValidatedPaperSummary(BaseModel):
    authors: list[str] | None = Field(description="List of Names and Surnames of the authors")
    achieved_accuracy: float | None = Field(description="Achieved accuracy in percentage")

    @field_validator('authors')
    @classmethod
    def author_must_have_name_and_surname(cls, authors: list[str] | None):
        if authors is None:
            return authors
        for author in authors:
            if len(author.split()) < 2:
                raise ValueError("Author must have both name and surname")
        return authors
    
    @field_validator('achieved_accuracy')
    @classmethod
    def accuracy_must_be_between_0_and_1(cls, accuracy: float | None):
        if accuracy is None:
            return accuracy
        if not 0 <= accuracy <= 1:
            raise ValueError("Accuracy must be between 0 and 1")
        return accuracy

agent = Agent(model=MODEL, system_prompt="Extract relevant fields from the provided abstract.", output_type=ValidatedPaperSummary)
result = agent.run_sync(user_prompt=f"Abstract to parse: {abstract}", retries=5)
result.output

ValidatedPaperSummary(authors=['John <UNKNOWN>', 'Jane <UNKNOWN>'], achieved_accuracy=0.74)

In [30]:
pretty_print_trace(result)


[0] ModelRequest
    └─ SystemPromptPart: SystemPromptPart(content='Extract relevant fields from the provided abstract.', timestamp=datetime.datetime(2026, 5, 23, 14, 50, 52, 817817, tzinfo=datetime.timezone.utc))
    └─ UserPromptPart: UserPromptPart(content='Abstract to parse: In this work authored by John and Jane, we have achieved accuracy of 74%', timestamp=datetime.datetime(2026, 5, 23, 14, 50, 52, 817825, tzinfo=datetime.timez

[1] ModelResponse
    └─ ToolCallPart: ToolCallPart(tool_name='final_result', args='{"authors": ["John","Jane"], "achieved_accuracy": 74}', tool_call_id='toolu_bdrk_01KUdPLkZcr19mMkpHDCy57L')

[2] ModelRequest
    └─ RetryPromptPart: RetryPromptPart(content=[{'type': 'value_error', 'loc': ('authors',), 'msg': 'Value error, Author must have both name and surname', 'input': ['John', 'Jane']}, {'type': 'value_error', 'loc': ('achieve

[3] ModelResponse
    └─ ToolCallPart: ToolCallPart(tool_name='final_result', args='{"authors": ["<UNKNOWN>","<UNKNOWN>"], "

# Validators +

In [31]:
# Validators can also immediately fix the data or transform/convert them

class ValidatedPaperSummary(BaseModel):
    achieved_accuracy: float | None
    
    @field_validator('achieved_accuracy')
    @classmethod
    def accuracy_must_be_between_0_and_1(cls, value: float | None):
        if value is None:
            return value
        if 1 < value <= 100: # <- ADDED this condition
            value = value / 100 # Assuming the value is a percentage
        if not 0 <= value <= 1:
            raise ValueError("Accuracy must be between 0 and 1")
        return value # Here we could return a transformed value

In [ ]:
# Demo: the validator repairs percentages, and still rejects nonsense
print(ValidatedPaperSummary(achieved_accuracy=74).achieved_accuracy)    # 0.74  — repaired
print(ValidatedPaperSummary(achieved_accuracy=0.74).achieved_accuracy)  # 0.74  — already fine, untouched
try:
    ValidatedPaperSummary(achieved_accuracy=250)
except ValueError as e:
    print(f"Rejected: {e}")                                             # 250 can't be saved

In [32]:
# Validators can also validate the output as a whole
from typing_extensions import Self
from pydantic import model_validator

class SummationEquation(BaseModel):
    a: int
    b: int
    result: int

    @model_validator(mode='after')
    def check_sum(self) -> Self:
        if self.a + self.b != self.result:
            raise ValueError('a + b does not equal result')
        return self

In [ ]:
# Demo: model validator checks a + b == result across all three fields
print(SummationEquation(a=2, b=3, result=5))    # valid
try:
    SummationEquation(a=2, b=3, result=6)
except ValueError as e:
    print(f"Rejected: {e}")                      # 2 + 3 != 6

# Exercise

Given a set of papers and their abstracts, rank them based on the achieved accuracy on the ImageNet-1K benchmark.

**Note**: the abstracts represent the achieved accuracy in mixed format (e.g. 0.84, 84%, 84 percent, etc.). Your validator should parse the agent's output and convert the achieved accuracy to a float between 0 and 1, so that the provided ranking function works correctly.

In [ ]:
ABSTRACTS = [
    {
        "title": "ConvNeXt-V2 adapters improve visual classification under limited fine-tuning",
        "year": 2024,
        "abstract": (
            "Parameter-efficient fine-tuning has become increasingly important for "
            "adapting large vision models to new classification tasks. We introduce "
            "a lightweight adapter module for ConvNeXt-V2 and evaluate it on the "
            "standard ImageNet-1K, CIFAR-100, and CIFAR-10 benchmarks. The model "
            "achieved 0.84 accuracy on ImageNet-1K, 0.91 accuracy on CIFAR-100, "
            "and 0.98 accuracy on CIFAR-10. In binary out-of-distribution detection "
            "experiments derived from CIFAR-10-C, the same model reached 0.93 AUROC. "
            "These results suggest that adapter-based fine-tuning can preserve "
            "strong general-purpose visual representations while reducing the number "
            "of trainable parameters."
        ),
    },
    {
        "title": "Masked autoencoder pretraining improves robustness in image classification",
        "year": 2024,
        "abstract": (
            "We studied whether masked autoencoder pretraining improves downstream "
            "classification robustness across widely used computer-vision benchmarks. "
            "A ViT-B/16 model was pretrained with random patch masking and then "
            "fine-tuned on ImageNet-1K, CIFAR-100, and CIFAR-10. The model obtained "
            "83% accuracy on ImageNet-1K, 89% accuracy on CIFAR-100, and 97% "
            "accuracy on CIFAR-10. On corrupted-image variants from ImageNet-C and "
            "CIFAR-10-C, the model achieved 86% AUROC for distinguishing clean from "
            "corrupted samples. The findings indicate that self-supervised "
            "pretraining improves robustness without sacrificing standard "
            "classification performance."
        ),
    },
    {
        "title": "A compact ResNet with knowledge distillation narrows the accuracy gap",
        "year": 2025,
        "abstract": (
            "Small convolutional networks remain attractive for deployment on "
            "resource-constrained devices, but they often underperform larger "
            "architectures. We trained a compact ResNet using knowledge distillation "
            "from a high-capacity vision transformer and benchmarked it on "
            "ImageNet-1K, CIFAR-100, and CIFAR-10. The student model achieved "
            "0.76 accuracy on ImageNet-1K, 0.84 accuracy on CIFAR-100, and 0.95 "
            "accuracy on CIFAR-10. In an auxiliary one-vs-rest evaluation on "
            "CIFAR-100 superclass labels, the model reached 0.88 AUROC. These "
            "results show that distillation can substantially improve small-model "
            "performance while retaining efficient inference."
        ),
    },
    {
        "title": "Data augmentation policies improve cross-dataset generalisation",
        "year": 2025,
        "abstract": (
            "Automated data augmentation can improve generalisation, but its effects "
            "vary across datasets and architectures. We evaluated a learned "
            "augmentation policy using a DeiT-S classifier trained and tested on "
            "ImageNet-1K, CIFAR-100, and CIFAR-10. The model reached 82% accuracy "
            "on ImageNet-1K, 88% accuracy on CIFAR-100, and 96% accuracy on "
            "CIFAR-10. When evaluated for confidence-based failure detection, it "
            "reported 85% AUROC on ImageNet-1K validation predictions and 90% "
            "AUROC on CIFAR-100 validation predictions. The augmentation policy "
            "was most beneficial for medium-sized training regimes and improved "
            "calibration as well as top-1 classification accuracy."
        ),
    },
    {
        "title": "Graph neural networks for molecular property prediction using MoleculeNet",
        "year": 2025,
        "abstract": (
            "We developed a graph neural network for molecular property prediction "
            "using atom-level message passing and bond-aware attention. Unlike the "
            "image-classification studies, this work did not evaluate on ImageNet-1K, "
            "CIFAR-100, or CIFAR-10 because its task involved molecular graphs rather "
            "than natural images. Instead, the model was evaluated on MoleculeNet "
            "benchmarks including Tox21, ClinTox, and HIV. It achieved 78% accuracy "
            "on Tox21, 74% accuracy on ClinTox, and 81% accuracy on HIV. The model "
            "also reached 86% AUROC on Tox21 and 91% AUROC on HIV, indicating strong "
            "performance on binary molecular classification tasks."
        ),
    },
]


In [19]:
MODEL = OpenAIChatModel(
    'openai/gpt-4o-mini',
    provider=OpenAIProvider(
        base_url='https://openrouter.ai/api/v1',
        api_key=os.environ['OPENROUTER_API_KEY'],
        http_client=httpx.AsyncClient(timeout=60),
    ))

In [63]:
class PaperResult(BaseModel):
    paper_title : str = Field("Title of the paper")
    accuracy_on_ImageNet1K : float | None = Field("Accuracy of the paper on the ImageNet1K benchmark")

# TODO: add field validator on accuracy_on_ImageNet1K to normalize the value to a float between 0 and 1

class Papers(BaseModel):
    papers_results : list[PaperResult] = Field("List of papers and their results on the ImageNet1k benchmark")

agent = Agent(model=MODEL, 
              system_prompt="Extract relevant fields from the provided abstract.", 
              output_type=Papers)

result= agent.run_sync(user_prompt=f"Extract the title and the accuracy on the ImageNet1k dataset from the following abstracts: {ABSTRACTS}")

In [64]:
pretty_print_trace(result)


[0] ModelRequest
    └─ SystemPromptPart: SystemPromptPart(content='Extract relevant fields from the provided abstract.', timestamp=datetime.datetime(2026, 7, 2, 17, 14, 28, 569484, tzinfo=datetime.timezone.utc))
    └─ UserPromptPart: UserPromptPart(content="Extract the title and the accuracy on the ImageNet1k dataset from the following abstracts: [{'title': 'ConvNeXt-V2 adapters improve visual classification under limited fine-tuning', 'year': 2024, 'abstract': 'Parameter-efficient fine-tuning has become increasingly important for adapting large vision models to new classification tasks. We introduce a lightweight adapter module for ConvNeXt-V2 and evaluate it on the standard ImageNet-1K, CIFAR-100, and CIFAR-10 benchmarks. The model achieved 0.84 accuracy on ImageNet-1K, 0.91 accuracy on CIFAR-100, and 0.98 accuracy on CIFAR-10. In binary out-of-distribution detection experiments derived from CIFAR-10-C, the same model reached 0.93 AUROC. These results suggest that adapter-based fi

In [65]:
# Helper cell to pretty-print the output of the agent
def print_results(papers: Papers):
    print(f"Extracted {len(papers.papers_results)} papers\n" + "-" * 40)
    for p in papers.papers_results:
        print(f"{p.accuracy_on_ImageNet1K}  {p.paper_title}")

print_results(result.output)

Extracted 5 papers
----------------------------------------
0.84  ConvNeXt-V2 adapters improve visual classification under limited fine-tuning
83.0  Masked autoencoder pretraining improves robustness in image classification
0.76  A compact ResNet with knowledge distillation narrows the accuracy gap
82.0  Data augmentation policies improve cross-dataset generalisation
None  Graph neural networks for molecular property prediction using MoleculeNet


In [53]:
# Ranking function to find the best paper on the ImageNet-1K benchmark
def best_per_benchmark(papers: Papers):
    scores = [
        (p.paper_title, p.accuracy_on_ImageNet1K)
        for p in papers.papers_results
        if p.accuracy_on_ImageNet1K is not None #Skip papers without reported accuracy
    ]
    title, acc = max(scores, key=lambda pair: pair[1])
    print(f"ImageNet-1K: best is {acc:.2f} — {title}")

best_per_benchmark(result.output)


ImageNet-1K: best is 83.00 — Masked autoencoder pretraining improves robustness in image classification
